In [91]:
import csv
import sys
import os

import pandas as pd

In [92]:
cbc = pd.read_csv("../../data/processed/contemporary/contemporary_corpus.csv")

In [93]:
cbc.head()

,pmid,title,abstract,journal,year
0,42626385,Rationally Designed Zwitterionic Peptides Impr...,Nanoparticle drug delivery systems (NP DDS) ha...,ACS applied nano materials,2026.0
1,42500347,miR-30b-5p Promotes Medulloblastoma Cell Proli...,"Medulloblastoma (MB), a malignant tumour arisi...",Indian journal of clinical biochemistry : IJCB,2026.0
2,42500346,Occupational Exposure to Lead Increases Inflam...,Lead is a hazardous heavy metal. It has seriou...,Indian journal of clinical biochemistry : IJCB,2026.0
3,42500345,hsa-miR-3529-5p through <i>F2RL3</i> Regulatio...,The emergence of drug resistance in gastric ca...,Indian journal of clinical biochemistry : IJCB,2026.0
4,42500340,Likelihood of a Novel Pathogenic <i>LDLR</i> M...,Familial hypercholesterolemia (FH) is an autos...,Indian journal of clinical biochemistry : IJCB,2026.0


In [94]:
cbc_sample = cbc.sample(n=50, random_state=42)

In [97]:
cbc_sample.set_index("pmid").to_csv("../../data/processed/contemporary/cbc_sampled_abstracts.csv")

In [96]:
cbc_sample.head()

,pmid,title,abstract,journal,year
92,42625765,Molecular Detection of Virulence Genes and Mul...,<i>Streptococcus agalactiae</i> is one of the ...,Archives of Razi Institute,2025.0
93,42621305,N6-methyladenosine-modified circLPAR3 drives d...,Epigenetic mechanisms represented by N6-methyl...,Genes & diseases,2026.0
179,42632605,Benzene Metabolite-Targeted Gene ATF7IP2 Drive...,Benzene exposure is a recognized environmental...,Toxicology letters,2026.0
124,41415885,R183Q GNAQ Sturge-Weber syndrome Leptomeningea...,"Sturge-Weber syndrome (SWS), a rare neurovascu...",Journal of vascular anomalies,2024.0
261,42376533,Aggressive basal cell carcinoma in a non-facia...,Basal cell carcinomas (BCCs) in cats are gener...,Open veterinary journal,2025.0


In [87]:
def write_pubtator_input(df: pd.DataFrame, out_path: str) -> None:
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(f"{row['pmid']}|t|{row['title']}\n")
            f.write(f"{row['pmid']}|a|{row['abstract']}\n")
            f.write("\n")

write_pubtator_input(cbc_sample, "../../../aioner/AIONER/example/input/cbc_50_sample.pubtator")
print(f"Wrote {len(cbc_sample)} abstracts for AIONER tagging.")

Wrote 50 abstracts for AIONER tagging.


### ONLY run the following after AIONER extraction is saved to `../../aioner/`

In [88]:
def parse_pubtator(filepath):
    """Parses a PubTator file into a list of entity rows."""
    rows = []
    current_pmid = None
    current_title = ""
    current_abstract = ""

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue

            if "|t|" in line:
                current_pmid, _, current_title = line.split("|", 2)
            elif "|a|" in line:
                _, _, current_abstract = line.split("|", 2)
            else:
                parts = line.split("\t")
                if len(parts) >= 5:
                    pmid, start, end, entity_text, entity_type = parts[:5]
                    rows.append({
                        "pmid": pmid,
                        "text": entity_text,
                        "entity_type": entity_type,
                        "start": start,
                        "end": end,
                        # "Title": current_title,
                    })
    return rows


def write_csv(rows, out_path):
    fieldnames = ["pmid", "text", "entity_type", "start", "end"]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {len(rows)} entity rows to {out_path}")


if __name__ == "__main__":
    input_path = r"../../aioner/cbc_50_sample.PubTator"
    output_path = os.path.splitext(input_path)[0] + ".csv"

    rows = parse_pubtator(input_path)
    write_csv(rows, output_path)

Wrote 1253 entity rows to ../../aioner/cbc_50_sample.csv


In [ ]:
aioner_entities = pd.read_csv("../../aioner/cbc_50_sample.csv")
aioner_entities.to_excel("../../data/processed/contemporary/cbc_entities.xlsx")